In [0]:
# Week 9 - Power BI Dashboard Refinement and Validation

## Project
# EV Charge IQ Analytics - Team 17

## Week 9 Objective
# Refine the Week-8 Power BI report, validate Gold-source consistency,
# test Power BI filters and interactions, reconcile important Power BI
# values against Databricks Gold tables, and document evidence-backed
# business insights.

## Approved Gold Sources
# 1. default.gold_overall_daily_activity
# 2. default.gold_station_daily_activity

## Week 9 Boundary
# No new Bronze/Silver source is introduced.
# No streaming implementation is performed.
# The existing Week-8 Gold model remains the business-data source.

## Week 9 Starting Point
# Week 9 starts from the validated Week-8 Gold-to-Power BI hand-off.
# The Week-8 Power BI report is refined and validated rather than rebuilt from scratch.

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    MIN(activity_date) AS min_date,
    MAX(activity_date) AS max_date
FROM default.gold_overall_daily_activity;

row_count,min_date,max_date
90,2026-01-01,2026-03-31


In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT station_id) AS station_count,
    MIN(activity_date) AS min_date,
    MAX(activity_date) AS max_date
FROM default.gold_station_daily_activity;

row_count,station_count,min_date,max_date
15660,174,2026-01-01,2026-03-31


In [0]:
%sql
SELECT
    station_id,
    activity_date,
    COUNT(*) AS duplicate_count
FROM default.gold_station_daily_activity
GROUP BY
    station_id,
    activity_date
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC;

station_id,activity_date,duplicate_count


In [0]:
## Power BI Date Slicer Interaction Test

# - Slicer field: `gold_overall_daily_activity[activity_date]`
# - Test period: 2026-02-01 to 2026-02-28
# - Overall Gold visuals responded to the slicer:
  # - Total Session Count
  # - Total Energy kWh
  # - Estimated Revenue INR
  # - Daily EV Charging Sessions
# - Station Gold visuals did not respond:
  # - Station-wise Charging Sessions
  # - Sessions by City Band

### Interpretation
# The date slicer filters visuals owned by `gold_overall_daily_activity`.
# The station-level visuals remain independent because they are sourced from `gold_station_daily_activity`. No unsafe relationship was added between the two independent Gold summary tables.

In [0]:
%sql
SELECT
    SUM(total_session_count) AS february_total_sessions
FROM default.gold_overall_daily_activity
WHERE activity_date BETWEEN '2026-02-01' AND '2026-02-28';

february_total_sessions
89484


In [0]:
## Week 9 Filtered Reconciliation Evidence

### Total EV Charging Sessions

# - Power BI filter: `activity_date = 2026-02-01 to 2026-02-28`
# - Power BI Card: `89K`
# - Owning Gold table: `default.gold_overall_daily_activity`
# - Gold validation query result: `89,484`
# - Reconciliation status: PASS

# The Power BI Card value reconciles to the owning Gold table under the same February 2026 date filter. Power BI displays the value using thousands (`K`) notation.

In [0]:
%sql
SELECT
    activity_date,
    total_session_count,
    total_energy_kwh,
    estimated_revenue_inr
FROM default.gold_overall_daily_activity
ORDER BY total_session_count DESC
LIMIT 10;

activity_date,total_session_count,total_energy_kwh,estimated_revenue_inr
2026-01-01,3705,95026.41100000002,1431200.8305000046
2026-03-25,3237,81369.83199999951,1219796.8299999987
2026-02-27,3232,81427.54999999971,1220822.8915000004
2026-02-25,3231,82399.32599999943,1238688.426000003
2026-02-09,3231,81818.16399999957,1228054.9325
2026-01-30,3225,81152.91499999976,1210798.3920000016
2026-01-11,3224,81352.69099999945,1224759.2055000016
2026-03-03,3224,83167.23999999938,1249308.3050000027
2026-01-27,3223,81632.61099999934,1223796.0529999975
2026-01-08,3222,80289.96399999932,1204011.8010000011


In [0]:
%sql
SELECT
    activity_date,
    total_session_count,
    total_energy_kwh,
    estimated_revenue_inr
FROM default.gold_overall_daily_activity
ORDER BY total_session_count ASC
LIMIT 10;

activity_date,total_session_count,total_energy_kwh,estimated_revenue_inr
2026-03-31,1990,51071.093999999684,739181.2199999988
2026-01-02,3143,80322.04499999918,1192892.971500001
2026-03-02,3150,78766.89199999945,1187165.8375000004
2026-01-05,3153,80503.8339999993,1212471.4164999996
2026-02-12,3155,80750.36199999954,1212414.2710000018
2026-01-31,3157,81204.15299999916,1214688.9865000006
2026-02-28,3158,81407.90099999952,1226517.7694999997
2026-01-03,3160,80979.89199999945,1220277.0889999971
2026-03-05,3168,82262.68199999955,1231068.3690000025
2026-03-23,3168,78347.13799999935,1172683.8560000006


In [0]:
%sql
SELECT
    activity_date,
    total_session_count,
    total_energy_kwh,
    estimated_revenue_inr
FROM default.gold_overall_daily_activity
WHERE activity_date IN ('2026-01-01', '2026-03-31')
ORDER BY activity_date;

activity_date,total_session_count,total_energy_kwh,estimated_revenue_inr
2026-01-01,3705,95026.41100000002,1431200.8305000046
2026-03-31,1990,51071.093999999684,739181.2199999988
